# Part I - Creating a Class

We are going to create our own class called SparkDataCheck that works on Spark SQL style data frames.

Create a .py file.

First import modules needed:

In [6]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import pyspark.pandas as ps

## Create a class Called SparkDataCheck

- Start a class called **SparkDataCheck**
- Create an `__init__` function that takes in self and a dataframe argument

  – Within this, create a `.df` attribute that is the dataframe

Let's start by creating our spark session

In [19]:
from pyspark.sql import SparkSession         # import the SparkSession class from PySpark#
spark = SparkSession.builder.getOrCreate()   # create or retrieve a SparkSession

Load a sample dataset from testing.

In [6]:
pdf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/red-wine.csv", delimiter = ";")
pdf.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


### Initiate a class and two classmethods

Initiate a class and create two @classmethods:

– One that creates an instance while reading in a csv file.\
   ∗ The method should have arguments for the class, the spark session, and the path to the file\
   ∗ You should use the `spark.read.load()` function as we did in our `pyspark` notebook. \
   ∗ Create an object of our class that is returned 
   
– One that creates an instance from a `pandas` dataframe (standard `pandas`)\
  ∗ The method should have arguments for the class, the spark session, and the pandas dataframe\
  ∗ You should use the `spark.CreateDataFrame()` function as we did in our `pyspark` notebook.\
  ∗ Create an object of our class that is returned

In [1]:
"""
SparkDataCheck.py

This module define a class that works 
on Spark SQL style data frames.
"""
# import modules needed
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

class SparkDataCheck:
    def __init__(self, df: DataFrame):
        # create a .df attribute
        self.df = df
            
  #=================================================================
    # Classmethod 1: create instance by reading a CSV file
    @classmethod   
    def from_csv(cls, spark, path):
        df = (spark.read
                   .format("csv")
                   .option("header", True)
                   .option("inferSchema", True)
                   .option("sep", sep)
                   .load(path))
        return cls(df)
   
  
    #==============================================================
    # Classmethod 2: create instance from a pandas DataFrame
    @classmethod
    def from_pandas(cls,spark, pandas_df):
        df = spark.createDataFrame(pandas_df)
        return cls(df)
    
         
    

### Test the classmethods

In [49]:
import importlib
import ST554_project2_part1
importlib.reload(ST554_project2_part1)

<module 'ST554_project2_part1' from '/home/jupyter-hfang4@ncsu.edu/ST-554-Project-2/ST554_project2_part1.py'>

In [50]:
# check with the red-wine.csv data
check_df = ST554_project2_part1.SparkDataCheck.from_csv(spark, "red-wine.csv")
check_df.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

In [51]:
import pandas as pd
pdf = pd.read_csv("red-wine.csv")
check_df2 = ST554_project2_part1.SparkDataCheck.from_pandas(spark, pdf)
check_df2.df.show(5)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

This shows that the two classmethods are working. 

### Create validation methods

- Create a couple of validation methods. Each validation method will **modify the df attribute of the object** using different column functions and **return itself (the object that
has the df attribute)** so that we can chain commands. We won’t do any of the returning of the data within our methods (such as `take()` or `collect()`). We’ll leave that as something the user can do
themselves! (** This will need to be done via something like `my_object.df.show()`).

#### Create Boolean column based on numeric bounds

– Create a method that checks if each value in a numeric column is within user defined limits (upper and lower bounds, inclusive) and returns the dataframe with an appended column of
Boolean values.

∗ The function should allow the user to supply a single column and a `lower` and `upper` value. Check that at least one of lower or upper is provided (if not provided, don’t check that side).\
∗ For any `NULL` values, return `NULL`\
∗ If the user supplies a non-numeric column (not float, int, longint, bigint, double, or integer), print a message and return the df without modification.\
∗ Hints: Check out the `.dtypes` attribute of the data frame. On a column, you can use the `.between()` method

In [1]:
#============================================
# 1. Validation methods
#============================================

# 1.1 create boolean column based on numeric bounds
from pyspark.sql.functions import col as spark_col

def check_numeric_range(self, col: str, lower: float = None, upper: float = None):
    """
    Append a Boolean column indicating whether values in a numeric column
    fall within user-defined lower and/or upper bounds (inclusive).
    NULL values remain NULL.
    Modifies self.df and returns self for method chaining.
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self

    # -----------------------------------
    # check if the columin is numeric
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, NumericType):
        print(f"Column '{col}' is not numeric.")
        return self

    #--------------------------------------
    # ensure at least one bound is provided
    #--------------------------------------
    if lower is None and upper is None:
        print("No bounds provided. Please provide at least one bound.")
        return self

    #--------------------------------------------------------
    # build the Boolean condition
    # Spark automatically returns NULL when the input is NULL
    #--------------------------------------------------------
    if lower is not None and upper is not None:
        # use Spark's between() when both bounds exist
        condition = spark_col(col).between(lower, upper)
    elif lower is not None:
        # only lower bound provided
        condition = spark_col(col) >= lower
    elif upper is not None:
        # only upper bound provided
        condition = spark_col(col) <= upper

    #---------------------------------
    # append Boolean column to dataframe
    #---------------------------------
    new_col_name = f"{col}_in_range"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
    

#### Create a method checking string columns

– Create a method that checks if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values.
∗ For any `NULL` values, return `NUL`\
∗ If the user supplies a non-string column print a message and return the df without modification\
∗ Hint: The `.isin()` method on a column is useful!\

In [1]:
# 1.2 create a method checking values fall within a set of levels
# import modules needed
from pyspark.sql.functions import col as spark_col
from pyspark.sql.types import StringType

def check_value_levels(self, col: str, levels):
    """
    Check whether values in a string column fall within 
    a user-specified set of allowed levels. 
    Appends a Boolean column. NULL values remain NULL.
    Modifies self.df and returns self for method chaining. 
    """
    # -----------------------------------
    # check if the column exists
    # -----------------------------------
    if col not in self.df.columns:
        print(f"Column '{col}' does not exist.")
        return self
    
    # -----------------------------------
    # check if the columin is string
    #------------------------------------
    dtype = self.df.schema[col].dataType
    if not isinstance(dtype, StringType):
        print(f"Column '{col}' is not a string column.")
        return self
    
   
    # build the Boolean condition
    condition = spark_col(col).isin(levels)
    # append the new Boolean column
    new_col_name = f"{col}_in_levels"
    self.df = self.df.withColumn(new_col_name, condition)
    return self
        
    

#### Create a method checking missiing values

– Create a method that checks if a each value in a column is missing (`NULL` specifically) and returns the dataframe with an appended column of Boolean values.
∗ Hint: The `.isNULL()` method on a column is useful!

### Create summarization methods

- Create a couple of summarization methods (this will generally be writing our own way to use functions that exist!). Each summarization method will return the summarizations
of the data (not an amended version of the dataframe) as a `pandas` data frame (regular `pandas` not `pandas-on-spark`).

#### Create a method to report min and max of a variable

– Create a method to report the `min` and `max` of a numeric column supplied by the user. Add an optional grouping variable (only one grouping variable allowed for simplicity).\
  ∗ The method should check if the column is numeric. If so, it should report the min and max of the column (grouped if appropriate). If not, a message should be printed that the column isn’t numeric and `None` should be returned\
  ∗ If no column is supplied, the method should report the `min` and `max` of any numeric columns (and produce no messages otherwise), grouped if appropriate\
  ∗ Hints: This part got a bit complicated but I used the min and max functions from `pyspark.sql.functions` and the `.agg()` method on a `.groupBy()` spark SQL style data frame (similar to the notes). For the grouped option with all numeric columns, I used `reduce()` from `functools` with `pd.merge()` to simplify the result into a single data frame.

In [3]:
#====================================
# 2. Summarization methods
#====================================

# -----------------------------------------------------------------------------------
# 2.1 define a method to report min and max of a numeric coulumn supplied by the user
# -----------------------------------------------------------------------------------
# import modules needed
from pyspark.sql.functions import min, max  
from pyspark.sql.types import NumericType   
from functools import reduce
import pandas as pd

def min_max(self, col = None, group = None):
    """
    Report min and max for:
    1. a user-supplied numeric column (grouped if provided)
    2. or all numeric columns if no column is supplied (grouped if provided)
    """
    #-------------------------------------
    # scenario 1: User supplies a column
    #-------------------------------------
    if col is not None:
        # check column exists
        if col not in self.df.columns:
            print(f"Column {col} don't exist.")
            return None
        
        # get the data type of the column
        dtype = self.df.schema[col].dataType

        # check if column is numeric
        if not isinstance(dtype, NumericType):
            print(f"Column '{col}' is not numeric.")
            return None
        
        # grouped version for a single numeric column
        if group is not None:
            return(
                self.df.groupBy(group)
                       .agg(min(col).alias(f"{col}_min"),
                           max(col).alias(f"{col}_max"))
            )
        
        # ungrouped version for a single numeric column      
        return self.df.select(
            min(col).alias(f"{col}_min"),
            max(col).alias(f"{col}_max")
        )
    
    #----------------------------------------------------------------
    # scenario 2: no column supplied: compute for all numeric columns
    #----------------------------------------------------------------
    # identify all numeric columns
    numeric_cols = [
        field.name
        for field in self.df.schema.fields
        if isinstance(field.dataType, NumericType)]
    
    # if no numeric columns exist, print a message
    if not numeric_cols:
        print("No numeric columns found.")
        return None
    
    # grouped version for all numeric columns
    if group is not None:
        # setup an enpty list
        dfs = []
        # compute grouped min/max for each numeric column 
        for c in numeric_cols:
            df_c = (
                self.df.groupBy(group)
                       .agg(min(c).alias(f"{c}_min"),
                           max(c).alias(f"{c}_max"))
            ).toPandas()
            dfs.append(df_c)
            
        # merge all grouped results into on DataFrame
        merged = reduce(lambda left, right: pd.merge(left, right, on = group),dfs)
        return merged
    
    # ungrouped scenartion for all numeric columns
    agg_exprs = []
    for c in numeric_cols:
        agg_exprs.extend([min(c).alisa(f"{c}_min"),
                         max(c).alisa(f"{c}_max")])
    # Return a DataFrame with all min/max values
    return self.df.select(*agg_exprs)

#### Create a method to report counts of string columns

– Create a method to report the counts associated with one or two string columns. Have the function take in two separate arguments for columns, with the second being optional and the first
required.\
    ∗ The method should check if the column(s) are strings. If so, it should report the counts for the combinations of levels of each variable or of the single variable. If not, a message should
be printed that the column is numeric.

## Check the created class with real data

Now we’ll use your class on some data! Create a .ipynb on the JupyterHub (use this same file for the steps
below and part II)
- In the notebook, provide an introduction and narrative to what you are about to do!
- Import your script so you have access to your class!
- Read in the air quality data we used in the first project>. I’ve downloaded this data as a .csv file
and it is available at https://www4.stat.ncsu.edu/online/datasets/air.csv. Use your method
that creates an instance of the class from this csv file.
- Provide 4-5 examples of using each of your methods on this object. Show some examples where the
messages need to print out, where only one bound is provided, etc.
- Now, read that same data set in using pandas (not pandas-on-spark). Use your method to create an
instance of this class from the pandas data frame.
- Provide 1 example method call on that object.